In [ ]:
# 1. Instalacja wymaganych bibliotek
!pip install geopandas geopy folium

# 2. Skrypt geokodujący
import pandas as pd
import geopandas as gpd
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
import folium
import re
import time

# Wczytanie pliku CSV
try:
    df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/nowe/Grycan.csv', header=None, names=['adres'], sep='\t')
except Exception as e:
    print(f"Standardowe wczytanie nie powiodło się, próba alternatywna... błąd: {e}")
    with open('/content/drive/MyDrive/Colab Notebooks/nowe/Grycan.csv', 'r', encoding='utf-8') as f:
        lines = [line.strip() for line in f if line.strip()]
    df = pd.DataFrame(lines, columns=['adres'])

# Inteligentna funkcja standaryzacji adresu dla Nominatim
def wyczysc_adres_zaawansowany(adres):
    # 1. Usunięcie dopisków z myślnikami (np. galerii handlowych typu "Wroclavia", "Pasaż Grunwaldzki")
    text = adres.split('-')[0].strip()

    # 2. Usunięcie przedrostków "ul." lub "ulica" na samym początku, aby nie mylić placów i alei
    text = re.sub(r'^(ul\.\s*|ulica\s*)', '', text, flags=re.IGNORECASE).strip()

    # 3. Usunięcie kodu pocztowego (np. 50-086 lub 55-040)
    text = re.sub(r'\d{2}-\d{3}', '', text).strip()

    # 4. Uproszczenie zakresów numerów budynków np. "5-22" na "5", "6-8" na "6"
    text = re.sub(r'(\d+)-\d+', r'\1', text)

    # 5. Usunięcie zbędnych spacji i przecinków
    text = re.sub(r'\s+', ' ', text).strip()
    text = text.strip(', ')

    # 6. Określenie miasta docelowego i przygotowanie czystego zapytania
    if 'bielany' in text.lower():
        if 'bielany wrocławskie' not in text.lower():
            query = f"{text}, Bielany Wrocławskie, Polska"
        else:
            query = f"{text}, Polska"
    else:
        if 'wrocław' not in text.lower() and 'wroclaw' not in text.lower():
            query = f"{text}, Wrocław, Polska"
        else:
            if 'polska' not in text.lower():
                query = f"{text}, Polska"
            else:
                query = text

    return query

df['adres_czysty'] = df['adres'].apply(wyczysc_adres_zaawansowany)

# Inicjalizacja geokodera
geolocator = Nominatim(user_agent="analiza_przestrzenna_grycan_wroclaw_v4", timeout=15)

print("Rozpoczynam geokodowanie...")

szerokosci = []
dlugosci = []
wyniki = []

# Proces geokodowania pętli z własnym mechanizmem rezerwowym (fallback)
for idx, row in df.iterrows():
    query = row['adres_czysty']
    loc = None
    try:
        # Pierwsza próba: pełny oczyszczony adres
        loc = geolocator.geocode(query)
        time.sleep(1.2)

        # Druga próba (fallback): Jeśli nie znalazło, spróbujmy uprościć np. usunąć dzielnicę/dodatkowe informacje po przecinku
        if not loc and ',' in query:
            uproszczony_query = query.split(',')[0].strip()
            miasto = "Bielany Wrocławskie, Polska" if "Bielany" in query else "Wrocław, Polska"
            loc = geolocator.geocode(f"{uproszczony_query}, {miasto}")
            time.sleep(1.2)

        # Trzecia próba (fallback): Sama ulica i miasto (bez numeru budynku)
        if not loc:
            match = re.search(r'^([^,0-9]+)', query)
            if match:
                sam_adres = match.group(1).strip()
                miasto = "Bielany Wrocławskie, Polska" if "Bielany" in query else "Wrocław, Polska"
                fallback_query = f"{sam_adres}, {miasto}"
                loc = geolocator.geocode(fallback_query)
                time.sleep(1.2)
    except Exception as e:
        print(f"Błąd przy geokodowaniu adresu: '{query}': {e}")

    wyniki.append(loc)
    if loc:
        szerokosci.append(loc.latitude)
        dlugosci.append(loc.longitude)
        print(f"Sukces: '{row['adres']}' -> [{loc.latitude}, {loc.longitude}]")
    else:
        szerokosci.append(None)
        dlugosci.append(None)
        print(f"Niepowodzenie: '{row['adres']}' (Zapytanie: '{query}')")

df['wynik'] = wyniki
df['szerokosc'] = szerokosci
df['dlugosc'] = dlugosci

# Odrzucenie pustych wyników
df_clean = df.dropna(subset=['szerokosc', 'dlugosc']).copy()
print(f"\nZakończono! Zgeokodowano pomyślnie: {len(df_clean)} z {len(df)} punktów.")

# Konwersja na warstwę przestrzenną (GeoDataFrame)
gdf = gpd.GeoDataFrame(
    df_clean,
    geometry=gpd.points_from_xy(df_clean.dlugosc, df_clean.szerokosc),
    crs="EPSG:4326"
)

# Zapis do formatu Shapefile
gdf_to_save = gdf.rename(columns={'adres_czysty': 'adr_czyst', 'szerokosc': 'szerokosc', 'dlugosc': 'dlugosc'})
gdf_to_save.to_file('grycan_punkty.shp')
print("Pliki Shapefile zostały zapisane pomyślnie.")

# Interaktywna mapa do podglądu
if not gdf.empty:
    m = folium.Map(location=[gdf['szerokosc'].mean(), gdf['dlugosc'].mean()], zoom_start=12, tiles='CartoDB positron')
    for idx, row in gdf.iterrows():
        folium.CircleMarker(
            location=[row['szerokosc'], row['dlugosc']],
            radius=6, popup=row['adres'], color='#5A3A31', fill=True, fill_color='#8B5A2B'
        ).add_to(m)
    display(m)

Rozpoczynam geokodowanie...
Sukces: 'Sucha 1, 50-086 Wrocław - Wroclavia' -> [51.0964161, 17.034822]
Sukces: 'plac Teatralny 6-8, 50-051 Stare Miasto, Wrocław, Polska' -> [51.1057522, 17.0323534]
Sukces: 'Plac Grunwaldzki 22, 50-363 Wrocław' -> [51.1125754, 17.0593128]
Sukces: 'ul. Krzywoustego 26, 51-421 Wrocław' -> [51.1376919, 17.0716031]
Sukces: 'CH Aleja Bielany, Czekoladowa 7-9, 55-040 Bielany Wrocławskie' -> [51.045823, 16.9598791]

Zakończono! Zgeokodowano pomyślnie: 5 z 5 punktów.
Pliki Shapefile zostały zapisane pomyślnie.


In [ ]:
import zipfile
import os
from google.colab import files

# Nazwa pliku ZIP
zip_filename = 'grycan_shapefile.zip'

# Lista plików wchodzących w skład warstwy Shapefile
shapefile_extensions = ['.shp', '.shx', '.dbf', '.prj', '.cpg']
files_to_zip = [f'grycan_punkty{ext}' for ext in shapefile_extensions]

# Pakowanie plików
with zipfile.ZipFile(zip_filename, 'w') as zipf:
    for file in files_to_zip:
        if os.path.exists(file):
            zipf.write(file)
            print(f"Dodano do archiwum: {file}")

# Pobranie spakowanego archiwum na komputer
print("Uruchamiam pobieranie pliku ZIP...")
files.download(zip_filename)

Dodano do archiwum: grycan_punkty.shp
Dodano do archiwum: grycan_punkty.shx
Dodano do archiwum: grycan_punkty.dbf
Dodano do archiwum: grycan_punkty.prj
Dodano do archiwum: grycan_punkty.cpg
Uruchamiam pobieranie pliku ZIP...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# 1. Reprojekcja GeoDataFrame do układu EPSG:2180 (PUWG 1992)
gdf_1992 = gdf.to_crs("EPSG:2180")

# 2. Przygotowanie kolumn (skrócenie nazw dla formatu DBF) i zapis do pliku
gdf_1992_save = gdf_1992.rename(columns={'adres_czysty': 'adr_czyst', 'szerokosc': 'szerokosc', 'dlugosc': 'dlugosc'})
gdf_1992_save.to_file('grycan_punkty_1992.shp')
print("Zapisano pomyślnie nową warstwę w układzie PUWG 1992 (EPSG:2180)!")

# 3. Spakowanie nowej warstwy do archiwum ZIP i automatyczne pobranie
import zipfile
import os
from google.colab import files

zip_filename_1992 = 'grycan_shapefile_1992.zip'
shapefile_extensions = ['.shp', '.shx', '.dbf', '.prj', '.cpg']
files_to_zip_1992 = [f'grycan_punkty_1992{ext}' for ext in shapefile_extensions]

with zipfile.ZipFile(zip_filename_1992, 'w') as zipf:
    for file in files_to_zip_1992:
        if os.path.exists(file):
            zipf.write(file)
            print(f"Dodano do archiwum: {file}")

print("Uruchamiam pobieranie pliku ZIP z układem 1992...")
files.download(zip_filename_1992)

Zapisano pomyślnie nową warstwę w układzie PUWG 1992 (EPSG:2180)!
Dodano do archiwum: grycan_punkty_1992.shp
Dodano do archiwum: grycan_punkty_1992.shx
Dodano do archiwum: grycan_punkty_1992.dbf
Dodano do archiwum: grycan_punkty_1992.prj
Dodano do archiwum: grycan_punkty_1992.cpg
Uruchamiam pobieranie pliku ZIP z układem 1992...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import folium

# Tworzenie nowej interaktywnej mapy z inną paletą kolorów (np. czerwień i złoto)
if not gdf.empty:
    # Wyznaczenie środka mapy na podstawie punktów
    map_center = [gdf['szerokosc'].mean(), gdf['dlugosc'].mean()]
    m_new = folium.Map(location=map_center, zoom_start=12, tiles='CartoDB positron')

    for idx, row in gdf.iterrows():
        folium.CircleMarker(
            location=[row['szerokosc'], row['dlugosc']],
            radius=8,
            popup=folium.Popup(f"<b>{row['adres']}</b>", max_width=300),
            color='#D32F2F',       # Żywy czerwony kolor obramowania
            fill=True,
            fill_color='#FFD700',  # Ciepły złoty kolor wypełnienia
            fill_opacity=0.8,
            weight=2
        ).add_to(m_new)

    print("Wygenerowano nową mapę w nowej palecie kolorystycznej:")
    display(m_new)
else:
    print("Brak danych przestrzennych do wyświetlenia.")

Wygenerowano nową mapę w nowej palecie kolorystycznej:


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Geokodowanie kawiarni

In [ ]:
import requests
import pandas as pd
import geopandas as gpd
import os
import shutil
from google.colab import files

# Definiujemy zapytanie Overpass QL z jawnym timeoutem
overpass_query = """
[out:json][timeout:25];
area["name"="Wrocław"]["admin_level"="8"]->.searchArea;
(
  node["amenity"="cafe"](area.searchArea);
  way["amenity"="cafe"](area.searchArea);
  relation["amenity"="cafe"](area.searchArea);
);
out center;
"""

# Lista różnych stabilnych serwerów lustrzanych Overpass API
overpass_servers = [
    "https://overpass.kumi.systems/api/interpreter",
    "https://lz4.overpass-api.de/api/interpreter",
    "https://z.overpass-api.de/api/interpreter",
    "https://overpass-api.de/api/interpreter"
]

# Kluczowe nagłówki: unikalny User-Agent chroni przed błędem 406
headers = {
    'User-Agent': 'GeocodingGrycanProjectWroclaw_v5_OSM_Student (student.gis.pwr@outlook.com)',
    'Content-Type': 'application/x-www-form-urlencoded; charset=UTF-8'
}

In [ ]:
overpass_query = """
[out:json][timeout:25];
area["name"="Wrocław"]["admin_level"="8"]->.wroclawArea;
area["name"="Bielany Wrocławskie"]->.bielanyArea;
(
  node["amenity"="cafe"](area.wroclawArea);
  way["amenity"="cafe"](area.wroclawArea);
  relation["amenity"="cafe"](area.wroclawArea);

  node["amenity"="cafe"](area.bielanyArea);
  way["amenity"="cafe"](area.bielanyArea);
  relation["amenity"="cafe"](area.bielanyArea);
);
out center;
"""

data = None
for url in overpass_servers:
    print(f"Próba pobrania danych metodą POST z serwera: {url}...")
    try:
        # Wysyłamy zapytanie w ciele POST (x-www-form-urlencoded)
        response = requests.post(url, data={'data': overpass_query}, headers=headers, timeout=25)
        if response.status_code == 200:
            data = response.json()
            print(f"Sukces! Dane pobrane pomyślnie z serwera: {url}")
            break
        else:
            print(f"Serwer zwrócił kod statusu {response.status_code}")
    except Exception as e:
        print(f"Błąd podczas łączenia z {url}: {e}")

if data and 'elements' in data:
    kawiarnie = []
    for element in data['elements']:
        tags = element.get('tags', {})

        # Współrzędne dla punktu, środka linii lub relacji
        lat = element.get('lat') or element.get('center', {}).get('lat')
        lon = element.get('lon') or element.get('center', {}).get('lon')

        ulica = tags.get('addr:street', '')
        numer = tags.get('addr:housenumber', '')
        adres_pelny = f"{ulica} {numer}".strip()

        kawiarnie.append({
            'nazwa': tags.get('name', 'Brak nazwy'),
            'adres': adres_pelny if adres_pelny else 'Brak adresu w OSM',
            'marka': tags.get('brand', 'Brak marki'),
            'lat': lat,
            'lon': lon
        })

    # Konwersja do DataFrame
    df_osm = pd.DataFrame(kawiarnie)
    df_osm_clean = df_osm.dropna(subset=['lat', 'lon'])

    if not df_osm_clean.empty:
        print(f"Przetworzono {len(df_osm_clean)} kawiarni we Wrocławiu i Bielanach Wrocławskich.")

        # Tworzenie GeoDataFrame (WGS84)
        gdf_kawiarnie = gpd.GeoDataFrame(
            df_osm_clean,
            geometry=gpd.points_from_xy(df_osm_clean.lon, df_osm_clean.lat),
            crs="EPSG:4326"
        )

        # Tworzenie katalogu wyjściowego do zapisu Shapefile
        folder_wyjsciowy = 'kawiarnie_wroclaw_shp'
        os.makedirs(folder_wyjsciowy, exist_ok=True)

        # Skrócenie nazw kolumn do maks. 10 znaków (standard plików DBF w Shapefile)
        gdf_save = gdf_kawiarnie.rename(columns={
            'nazwa': 'name_osm',
            'adres': 'address_osm',
            'marka': 'brand_osm'
        })

        # Zapis do Shapefile
        gdf_save.to_file(f"{folder_wyjsciowy}/kawiarnie_wroclaw.shp")

        # Pakowanie do archiwum ZIP
        zip_name = 'kawiarnie_wroclaw_ArcMap'
        shutil.make_archive(zip_name, 'zip', folder_wyjsciowy)
        print(f"Pliki Shapefile zostały spakowane do archiwum {zip_name}.zip")

        # Automatyczne pobieranie pliku
        print("Rozpoczynam pobieranie spakowanego pliku na Twój komputer...")
        files.download(f"{zip_name}.zip")
    else:
        print("Pobrane dane nie zawierały poprawnych współrzędnych przestrzennych.")
else:
    print("Nie udało się uzyskać danych z żadnego serwera Overpass. Spróbuj ponownie za chwilę.")

Próba pobrania danych metodą POST z serwera: https://overpass.kumi.systems/api/interpreter...
Błąd podczas łączenia z https://overpass.kumi.systems/api/interpreter: HTTPSConnectionPool(host='overpass.kumi.systems', port=443): Read timed out. (read timeout=25)
Próba pobrania danych metodą POST z serwera: https://lz4.overpass-api.de/api/interpreter...
Sukces! Dane pobrane pomyślnie z serwera: https://lz4.overpass-api.de/api/interpreter
Przetworzono 325 kawiarni we Wrocławiu i Bielanach Wrocławskich.
Pliki Shapefile zostały spakowane do archiwum kawiarnie_wroclaw_ArcMap.zip
Rozpoczynam pobieranie spakowanego pliku na Twój komputer...


/tmp/ipykernel_907/2622446166.py:79: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  gdf_save.to_file(f"{folder_wyjsciowy}/kawiarnie_wroclaw.shp")
/usr/local/lib/python3.13/dist-packages/pyogrio/raw.py:733: RuntimeWarning: Normalized/laundered field name: 'address_osm' to 'address_os'
  ogr_write(


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>